In [3]:
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup

# Load keywords
df = pd.read_excel("Keyword.xlsx")
df.columns = df.columns.str.strip()
keywords = df['Keywords'].dropna().astype(str).tolist()

# Setup Chrome
chrome_options = Options()
chrome_options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=chrome_options)

all_data = []

def extract_tenders(html, keyword):
    soup = BeautifulSoup(html, "html.parser")
    tenders = soup.find_all("div", class_="txt")
    results = []

    for tender in tenders:
        try:
            # TDR number and description
            tdr_tag = tender.find("a", class_="m-brief")
            tdr_number = tdr_tag.find("span", class_="m-tender-id").get_text(strip=True)
            description = tdr_tag.get_text(strip=True)

            # State (3rd span in <strong> inside h2)
            header_spans = tender.find("h2", class_="workDesc").find_all("span")
            state = header_spans[-1].get_text(strip=True) if len(header_spans) >= 3 else ""

            # Due date parts
            month = tender.find("span", class_="month").get_text(strip=True)
            day = tender.find("span", class_="day").get_text(strip=True)
            year = tender.find("span", class_="year").get_text(strip=True)
            due_date = f"{month} {day}, {year}"

            # Tender Value
            value_tag = tender.find("span", class_="tender-value")
            tender_value = value_tag.get_text(strip=True).replace('\n', ' ') if value_tag else ""

            # Save
            results.append({
                "TDR Number": tdr_number,
                "Item Description": description,
                "State": state,
                "Due Date": due_date,
                "Tender Value": tender_value,
                "Keyword": keyword
            })
        except Exception as e:
            print(f"⚠️ Skipped a tender block due to error: {e}")
            continue

    return results

# Process all keywords
for keyword in keywords:
    print(f"\n🔍 Scraping for keyword: {keyword}")
    query = keyword.lower().replace(" ", "-")
    url = f"https://www.tenderdetail.com/Indian-tender/{query}-tenders"
    try:
        driver.get(url)
        time.sleep(4)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3)
        html = driver.page_source
        tenders = extract_tenders(html, keyword)
        all_data.extend(tenders)
    except Exception as e:
        print(f"❌ Failed on '{keyword}': {e}")

driver.quit()

# Save to Excel
if all_data:
    pd.DataFrame(all_data).to_excel("structured_keyword_tenders.xlsx", index=False)
    print("\n✅ Done! Data saved to 'structured_keyword_tenders.xlsx'")
else:
    print("\n⚠️ No data extracted.")



🔍 Scraping for keyword: UGPL

🔍 Scraping for keyword: Rising main

🔍 Scraping for keyword: water distribution systems

🔍 Scraping for keyword: Supplying  and laying DI pipes

🔍 Scraping for keyword: Supply of di pipes

🔍 Scraping for keyword: supply and delivery of DI pipe

🔍 Scraping for keyword: di k7

🔍 Scraping for keyword: di k9

🔍 Scraping for keyword: Megalift

🔍 Scraping for keyword: sewerage

🔍 Scraping for keyword: waste treatment

🔍 Scraping for keyword: irrigation

🔍 Scraping for keyword: distribution network

🔍 Scraping for keyword: distribution of water

🔍 Scraping for keyword: Gravity main

🔍 Scraping for keyword: Diversion network

🔍 Scraping for keyword: Water suppy scheme

🔍 Scraping for keyword: Distribution Main

🔍 Scraping for keyword: PHE

🔍 Scraping for keyword: Laying

🔍 Scraping for keyword: AMRUT

🔍 Scraping for keyword: JJM

🔍 Scraping for keyword: WRD

🔍 Scraping for keyword: DI PIPES

🔍 Scraping for keyword: OHR

🔍 Scraping for keyword: STP

🔍 Scraping for